# RSNA Knee — Notebook 05 FINAL: OOF-Selected Ensemble + Submission

This is the **final notebook**.

It does **no more training**. It uses:

1. the original 20-member DINO ensemble as the safety/base family;
2. Notebook 04 output for expert-CV evidence and the 5 FT-C checkpoints;
3. a conservative nested target-wise selector to decide whether any FT-C contribution
   is stable enough to blend with BASE;
4. final hidden-test inference;
5. exactly one competition file:

`/kaggle/working/submission.csv`

### Why BASE remains the anchor

Notebook 04 reported:

- BASE macro AUC: ~0.97897
- FT-A macro AUC: ~0.97237
- FT-B macro AUC: ~0.97204
- FT-C macro AUC: ~0.97204
- FT-C promoted: 0 / 5 (all five rolled back to FT-B)

So this notebook never blindly replaces BASE. A fine-tuned contribution is used only
for targets where a nested fold-wise blend shows a stable OOF ranking gain.

## Fixed V2 — final inference metadata contract repaired

The failed run stopped at `pick_slots()` with `KeyError: 'fatsat'`.

`walk_selected()` returns raw DICOM header rows. The successful pipeline always
passes those rows through `annotate(...)`, which creates the derived `fatsat`,
`fluid`, `weight`, and `px` columns required by the slot selector.

This version restores that exact preprocessing contract and adds preflight checks
so the same failure cannot silently recur.

## Required Kaggle inputs

Attach **only these four inputs**:

1. **RSNA Knee Abnormality Detection** competition
2. **DINO v2 — dinov2-small**
3. **RSNA Knee weights** — the original package containing the 20 base checkpoints
4. **Notebook 04: Target-Specialist Fine-Tuning** output (`rsna_stage4_ftc`)

You do **not** need Notebook 01, 02, or 03 as separate inputs.

In [ ]:
from __future__ import annotations

import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import json
import math
import re
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import roc_auc_score

T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

# Final inference controls.
EVAL_BATCH = int(os.environ.get("FINAL_EVAL_BATCH", "8"))
TIME_BUDGET = float(os.environ.get("FINAL_TIME_BUDGET_HOURS", "8")) * 3600
MIN_WINDOWS = int(os.environ.get("FINAL_MIN_WINDOWS", "3"))

# Conservative nested blend selector.
ALPHA_GRID = np.asarray([0.0, 0.10, 0.20, 0.30, 0.40, 0.50], dtype=float)
META_MIN_AUC_GAIN = float(os.environ.get("META_MIN_AUC_GAIN", "0.002"))
FULL_MIN_AUC_GAIN = float(os.environ.get("FULL_MIN_AUC_GAIN", "0.002"))
MIN_NONZERO_OUTER_CHOICES = int(os.environ.get("MIN_NONZERO_OUTER_CHOICES", "3"))

CROP_MM = 130.0
CACHE_IMG = 336
IMG = CACHE_IMG
GROUP = 3
CACHE_SLICES = 12
N_GROUP = max(CACHE_SLICES // GROUP, 1)
SLICE_BAND = (0.20, 0.80)

HDR_THREADS = int(os.environ.get("HDR_THREADS", "16"))
PIX_THREADS = int(os.environ.get("PIX_THREADS", "12"))
ORDER_THREADS = int(os.environ.get("ORDER_THREADS", "32"))
ORDER_BUDGET_S = 5400
LAT_MIN_OFFSET_MM = 20.0
LEGACY_LAT_OFFSET_MM = 5.0

RULES_NATIVE = {
    "order": "normal",
    "lat": "centre",
    "slot_fallback": False,
    "decode_fill": "nearest",
}
RULES_LEGACY = {
    "order": "dominant_axis",
    "lat": "corner_x",
    "slot_fallback": True,
    "decode_fill": "zero",
}
RULES = dict(RULES_NATIVE)

SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]
SLOTS = SLOTS_RECOVERED
N_SLOT = len(SLOTS)

POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(
    r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
    r"water excit|\btirm\b|\bsting\b|\bfatsup\b"
)
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")

FRONTIER_TARGET_POOL = {
    "Fracture": "max",
    "Contusion": "max",
    "Medial Meniscus": "max",
    "Lateral Meniscus": "max",
    "ACL": "top2",
    "MCL": "top2",
    "Baker's": "max",
}

FINAL_PATH = Path("/kaggle/working/submission.csv")
DIAG_DIR = Path("/kaggle/working/final_diagnostics")
DIAG_DIR.mkdir(parents=True, exist_ok=True)

# Never leave a stale submission from an earlier failed interactive run.
if FINAL_PATH.exists():
    FINAL_PATH.unlink()


## 1. Discover and validate the four required inputs

In [ ]:
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


class WeightsError(RuntimeError):
    pass


def find_root():
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
    ]
    for c in candidates:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c

    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            p = Path(root)
            if (p / "test.csv").is_file() and (p / "test_series").is_dir():
                return p
    raise FileNotFoundError("RSNA Knee competition mount not found")


def _looks_like_dinov2_config(config_path):
    try:
        cfg = json.loads(Path(config_path).read_text())
    except Exception:
        return False
    text = json.dumps(cfg).lower()
    return (
        cfg.get("model_type", "").lower() == "dinov2"
        or "dinov2" in text
        or any("dinov2" in str(x).lower() for x in cfg.get("architectures", []) or [])
    )


def find_dinov2():
    explicit = os.environ.get("KNEE_DINOV2_DIR", "").strip()
    if explicit:
        p = Path(explicit)
        if (p / "config.json").is_file():
            return p
        raise FileNotFoundError(f"KNEE_DINOV2_DIR={explicit!r} has no config.json")

    preferred = Path("/kaggle/input/models/metaresearch/dinov2/pytorch/small/1")
    if (preferred / "config.json").is_file():
        return preferred

    base = Path("/kaggle/input")
    hits = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "config.json" in files and _looks_like_dinov2_config(Path(root) / "config.json"):
                hits.append(Path(root))

    for p in hits:
        try:
            cfg = json.loads((p / "config.json").read_text())
            if int(cfg.get("hidden_size", -1)) == 384:
                return p
        except Exception:
            pass
    return hits[0] if hits else None


def _manifest_complete(root):
    root = Path(root)
    mp = root / "manifest.json"
    if not mp.is_file():
        return None
    try:
        man = json.loads(mp.read_text())
    except Exception:
        return None
    members = man.get("members")
    if not isinstance(members, list) or not members:
        return None
    if any(not (root / str(m.get("file"))).is_file() for m in members):
        return None
    return man


def find_stage4():
    explicit = os.environ.get("STAGE4_DIR", "").strip()
    if explicit:
        p = Path(explicit)
        st = json.loads((p / "stage4_status.json").read_text())
        man = _manifest_complete(p)
        if st.get("stage4_ok") and st.get("stage") == "FT-C_TARGET_SPECIALIST" and man:
            return p, st, man
        raise FileNotFoundError(f"STAGE4_DIR={explicit!r} is not a complete Stage-4 package")

    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "stage4_status.json" not in files:
                continue
            p = Path(root)
            try:
                st = json.loads((p / "stage4_status.json").read_text())
            except Exception:
                continue
            man = _manifest_complete(p)
            if st.get("stage4_ok") and st.get("stage") == "FT-C_TARGET_SPECIALIST" and man:
                return p, st, man
    raise FileNotFoundError("Notebook 04 output (rsna_stage4_ftc) not found")


def find_base_weights(stage4_path):
    explicit = os.environ.get("KNEE_WEIGHTS_DIR", "").strip()
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    candidates.append(Path("/kaggle/input/rsna-knee-weights"))

    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "manifest.json" in files:
                candidates.append(Path(root))

    seen = set()
    for p in candidates:
        key = str(p)
        if key in seen or p == stage4_path:
            continue
        seen.add(key)
        man = _manifest_complete(p)
        if man is None:
            continue

        # The original safety family is the 20-member package.
        if len(man.get("members", [])) >= 10:
            return p, man

    raise FileNotFoundError(
        "Original RSNA Knee weights package not found. "
        "Attach the 'RSNA Knee weights' dataset containing the 20 base checkpoints."
    )


ROOT = find_root()
DINO = find_dinov2()
STAGE4, STAGE4_STATUS, STAGE4_MANIFEST = find_stage4()
BASE_WEIGHTS, BASE_MANIFEST = find_base_weights(STAGE4)

if DINO is None:
    raise FileNotFoundError("DINOv2-small model input not found")

log(f"competition: {ROOT}")
log(f"DINOv2-small: {DINO}")
log(f"original base weights: {BASE_WEIGHTS} ({len(BASE_MANIFEST['members'])} members)")
log(f"Stage 4: {STAGE4} ({len(STAGE4_MANIFEST['members'])} members)")

if len(BASE_MANIFEST["members"]) != 20:
    print(
        "WARNING: expected 20 original base members, found",
        len(BASE_MANIFEST["members"]),
        "- continuing because the package is complete."
    )

display(pd.DataFrame([{
    "stage4_ok": STAGE4_STATUS.get("stage4_ok"),
    "stage4_promoted": STAGE4_STATUS.get("n_promoted"),
    "stage4_rollback": STAGE4_STATUS.get("n_rolled_back_to_ftb"),
    "stage4_best_single": STAGE4_STATUS.get("best_single_cv_branch_by_macro_auc"),
}]))

## 2. Nested OOF selector — BASE remains default

In [ ]:
oof_path = STAGE4 / "expert_cv_predictions_stage4.csv"
summary_path = STAGE4 / "expert_cv_branch_summary_stage4.csv"

oof = pd.read_csv(oof_path, dtype={"StudyInstanceUID": str})
branch_summary = pd.read_csv(summary_path)

required = {"StudyInstanceUID", "ft_fold", *TARGETS}
for t in TARGETS:
    required.add(f"base__{t}")
    required.add(f"ftc__{t}")
missing = required.difference(oof.columns)
if missing:
    raise RuntimeError(f"Stage-4 OOF is missing columns: {sorted(missing)}")

display(branch_summary)


def pct_rank_1d(x):
    return pd.Series(np.asarray(x, float)).rank(
        method="average", pct=True
    ).to_numpy(np.float64)


def safe_auc(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    good = np.isfinite(y) & np.isfinite(p)
    y, p = y[good], p[good]
    if len(y) < 2 or len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, p))


def select_target_alpha(y, fold, base_prob, ftc_prob):
    # Rank features are built without labels. This mirrors the final test-space rank blend.
    rb = pct_rank_1d(base_prob)
    rf = pct_rank_1d(ftc_prob)
    folds = sorted(pd.Series(fold).dropna().astype(int).unique().tolist())

    meta_pred = np.full(len(y), np.nan, dtype=float)
    outer_choices = []

    for outer in folds:
        tr = np.asarray(fold) != outer
        va = np.asarray(fold) == outer

        train_scores = []
        for a in ALPHA_GRID:
            p = (1.0 - a) * rb[tr] + a * rf[tr]
            train_scores.append(safe_auc(np.asarray(y)[tr], p))

        # Highest train-fold AUC; ties always choose the smaller FT-C weight.
        finite = np.asarray([
            s if np.isfinite(s) else -np.inf for s in train_scores
        ])
        best_i = int(np.argmax(finite))
        a = float(ALPHA_GRID[best_i])
        outer_choices.append(a)
        meta_pred[va] = (1.0 - a) * rb[va] + a * rf[va]

    base_auc = safe_auc(y, rb)
    ftc_auc = safe_auc(y, rf)
    meta_auc = safe_auc(y, meta_pred)

    full_scores = []
    for a in ALPHA_GRID:
        full_scores.append(safe_auc(y, (1.0 - a) * rb + a * rf))
    finite_full = np.asarray([
        s if np.isfinite(s) else -np.inf for s in full_scores
    ])
    full_best_i = int(np.argmax(finite_full))
    full_best_alpha = float(ALPHA_GRID[full_best_i])
    full_best_auc = float(finite_full[full_best_i])

    nonzero = int(sum(a > 0 for a in outer_choices))
    median_alpha = float(np.median(outer_choices))
    # Snap the median back to our safe grid, preferring the nearest smaller alpha on ties.
    final_alpha = float(
        ALPHA_GRID[np.argmin(np.abs(ALPHA_GRID - median_alpha))]
    )

    meta_gain = meta_auc - base_auc if np.isfinite(meta_auc) and np.isfinite(base_auc) else -np.inf
    full_gain = full_best_auc - base_auc if np.isfinite(base_auc) else -np.inf

    promoted = (
        final_alpha > 0
        and nonzero >= MIN_NONZERO_OUTER_CHOICES
        and meta_gain >= META_MIN_AUC_GAIN
        and full_gain >= FULL_MIN_AUC_GAIN
    )

    if not promoted:
        final_alpha = 0.0

    return {
        "base_auc": base_auc,
        "ftc_auc": ftc_auc,
        "nested_meta_auc": meta_auc,
        "nested_meta_gain": meta_gain,
        "full_best_alpha": full_best_alpha,
        "full_best_auc": full_best_auc,
        "full_best_gain": full_gain,
        "outer_choices": outer_choices,
        "nonzero_outer_choices": nonzero,
        "final_alpha": final_alpha,
        "promoted": bool(promoted),
    }


selection_rows = []
FINAL_ALPHAS = {}

fold = oof["ft_fold"].to_numpy(int)

for t in TARGETS:
    y = oof[t].to_numpy(float)
    res = select_target_alpha(
        y,
        fold,
        oof[f"base__{t}"].to_numpy(float),
        oof[f"ftc__{t}"].to_numpy(float),
    )
    FINAL_ALPHAS[t] = float(res["final_alpha"])
    selection_rows.append({
        "target": t,
        "base_auc": res["base_auc"],
        "ftc_auc": res["ftc_auc"],
        "nested_meta_auc": res["nested_meta_auc"],
        "nested_meta_gain": res["nested_meta_gain"],
        "full_best_alpha": res["full_best_alpha"],
        "full_best_auc": res["full_best_auc"],
        "full_best_gain": res["full_best_gain"],
        "outer_choices": json.dumps(res["outer_choices"]),
        "nonzero_outer_choices": res["nonzero_outer_choices"],
        "final_alpha": res["final_alpha"],
        "promoted": res["promoted"],
    })

selection = pd.DataFrame(selection_rows)
selection.to_csv(DIAG_DIR / "oof_blend_selection.csv", index=False)

print("Final target-wise FT-C weights:")
display(selection)

print(json.dumps(FINAL_ALPHAS, indent=2))

(DIAG_DIR / "final_blend_config.json").write_text(json.dumps({
    "method": "nested expert-CV conservative BASE-to-FT-C rank blend",
    "alpha_grid": ALPHA_GRID.tolist(),
    "meta_min_auc_gain": META_MIN_AUC_GAIN,
    "full_min_auc_gain": FULL_MIN_AUC_GAIN,
    "min_nonzero_outer_choices": MIN_NONZERO_OUTER_CHOICES,
    "alphas": FINAL_ALPHAS,
    "base_is_default": True,
}, indent=2))

NEED_FTC = any(a > 0 for a in FINAL_ALPHAS.values())
print("Need FT-C hidden-test inference:", NEED_FTC)
if not NEED_FTC:
    print("No target cleared the conservative gate: final hidden-test inference will use BASE only.")

## 3. Checkpoint pixel/preprocessing contract

In [ ]:
def slot_scheme_for_names(names):
    names = list(names)
    rec = [x[0] for x in SLOTS_RECOVERED]
    pub = [x[0] for x in SLOTS_PUBLIC]
    if names == rec:
        return SLOTS_RECOVERED
    if names == pub:
        return SLOTS_PUBLIC
    raise WeightsError(f"Unknown checkpoint slot scheme: {names}")


def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM
    global SLICE_BAND, RULES, SLOTS, N_SLOT

    CACHE_IMG = IMG = int(cfg["img"])
    GROUP = int(cfg["group"])
    CACHE_SLICES = int(cfg["slices"])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg["crop_mm"])
    SLICE_BAND = tuple(float(x) for x in cfg["band"])

    rules = cfg.get("rules") or RULES_NATIVE
    unknown = {
        k: v for k, v in rules.items()
        if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])
    }
    if unknown:
        raise WeightsError(f"unsupported pixel rules in manifest: {unknown}")
    RULES = {**RULES_NATIVE, **rules}

    SLOTS = slot_scheme_for_names(cfg["slots"])
    N_SLOT = len(SLOTS)

## 4. Exact DICOM preprocessing used by the successful pipeline

In [ ]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Position and orientation are read from the same header probe() already
            # opens, so they cost nothing, and they are what recovers the side when the
            # Laterality tag is absent - which it is for half the studies here.
            "ImagePositionPatient", "ImageOrientationPatient"]


def _hdr_vec(s, n):
    """Parse a DICOM multi-value string as stored by probe(): floats joined by `|`."""
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def side_from_geometry(h):
    """Study -> 'L' / 'R' / None, from where the image sits in the patient.

    `Laterality` (0020,0060) is Type 2C and may legitimately be absent; in this corpus it
    is missing on exactly half the studies, and the vendors it is missing from are whole
    vendors rather than scattered series. A study with no tag is not a left knee, but the
    normalisation upstream treats it as one, so half the corpus was never normalised and
    the five side-defined targets - the two menisci, the two tibiofemoral compartments
    and the medial collateral ligament - saw that axis reversed on a large minority of it.

    The patient coordinate system fixes this without the tag: +x is the patient's left, so
    the centre of a right knee sits at negative x. The centre is used rather than
    `ImagePositionPatient` itself because that is the corner of the image, which is offset
    by half a field of view - enough to change the sign on a knee near the midline.

    The median over a study's series is what is thresholded, not a single series: probe()
    reads one arbitrary slice per series, which on a sagittal stack can sit anywhere
    across the joint. Studies whose centre falls near the midline are left unresolved
    rather than guessed - measured against the tagged half, the rule is right 97% of the
    time overall and no better than chance inside 20 mm.
    """
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def side_from_corner_x(h):
    """The laterality an imported member was fitted under.

    It thresholds the median raw `ImagePositionPatient` x over a study's series. That is
    the x of the image *corner*, not of its centre, so it differs from the rule above by
    up to half a field of view - which is enough to reverse the sign on a knee scanned
    near the midline. The dead zone is 5 mm rather than 20 mm, so it also commits on
    studies the rule above leaves unresolved.

    Neither difference changes a shape. Each one decides whether a study is mirrored, and
    a study mirrored one way at training and the other at inference presents the five
    side-defined targets with their axis reversed.
    """
    out = {}
    for st, g in h.groupby("StudyInstanceUID"):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        # DICOM patient coordinates are LPS: +x is the patient's left.
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else ("R" if x < 0 else "L")
    return out


def lat_of(h, tag=""):
    """Study -> 'L' / 'R' / None: the tag where it exists, geometry where it does not.

    The tag is present on exactly half the studies here and is sometimes an empty
    string rather than absent, which is not the same as NaN. Treating the other half
    as left-sided is what `normalise_laterality` did by omission, so the geometry
    fallback is not a refinement - it is the difference between normalising half the
    corpus and normalising all of it.
    """
    geo = side_from_corner_x(h) if RULES["lat"] == "corner_x" else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        if RULES["lat"] == "corner_x" and "ImageLaterality" in g.columns:
            # The legacy rule reads the second tag too, so a study tagged only there is
            # resolved from the tag rather than from geometry.
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, "
        f"{n_none} unresolved; tag and geometry disagree on {n_disagree} "
        f"({n_disagree / max(n_tag, 1):.1%} of the tagged)")
    return d



def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    """Every series directory of a split, with one header read per series.

    An absent split returns an empty frame *with the columns annotate expects*. Returning
    a bare DataFrame looks like the same thing and is not: the next call indexes
    `SeriesDescription` and raises KeyError, so the branch that exists to survive a
    missing split is what turns it into a crash.
    """
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header."""
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # GE writes SAT_GEMS for spatial saturation, so ScanOptions must be matched as
    # exact tokens; a substring test on "SAT" fires on non-fat-sat series.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df

In [ ]:
def walk_selected(split, study_ids):
    study_ids = {str(x) for x in study_ids}
    base = ROOT / split
    items = []

    if not base.is_dir():
        raise FileNotFoundError(base)

    for study_id in sorted(study_ids):
        study_dir = base / study_id
        if not study_dir.is_dir():
            continue
        for series in os.scandir(study_dir):
            if series.is_dir():
                items.append((split, study_id, series.name, series.path))

    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))

    out = pd.DataFrame(rows)
    if out.empty:
        raise RuntimeError(f"No DICOM series found for {len(study_ids)} selected studies")
    return out

In [ ]:
def _as_bool(v):
    if pd.isna(v):
        return None
    s = str(v).strip().upper()
    if s in {"1", "TRUE", "T", "YES", "Y"}:
        return True
    if s in {"0", "FALSE", "F", "NO", "N"}:
        return False
    try:
        f = float(s)
        return True if f == 1 else False if f == 0 else None
    except Exception:
        return None


def audit_official_sequence_metadata(inferred, official):
    """Audit only. Do not change imported checkpoint pixels from this table."""
    need = {"SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"}
    if inferred.empty or official.empty or not need.issubset(official.columns):
        return
    a = inferred[["SeriesInstanceUID", "fluid", "fatsat"]].copy()
    b = official[["SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"]].copy()
    b["official_fluid"] = b["Fluid_Sensitive"].map(_as_bool)
    b["official_fatsat"] = b["Fat_Suppression"].map(_as_bool)
    m = a.merge(b[["SeriesInstanceUID", "official_fluid", "official_fatsat"]],
                on="SeriesInstanceUID", how="inner")
    for inferred_col, official_col, name in [
        ("fluid", "official_fluid", "Fluid_Sensitive"),
        ("fatsat", "official_fatsat", "Fat_Suppression"),
    ]:
        valid = m[official_col].notna() & m[inferred_col].notna()
        if valid.any():
            agree = (m.loc[valid, inferred_col].astype(bool).values ==
                     m.loc[valid, official_col].astype(bool).values).mean()
            log(f"metadata audit {name}: {agree:.1%} agreement on {int(valid.sum())} series")

In [ ]:
def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties are broken toward the stack with the most slices: a thicker stack samples the
    joint more densely, and the three-slice sampler below benefits from the margin.
    """
    required_meta = {
        "StudyInstanceUID", "SeriesInstanceUID", "n_slices",
        "fatsat", "fluid", "px",
    }
    missing_meta = required_meta.difference(series_df.columns)
    if missing_meta:
        raise RuntimeError(
            "pick_slots received raw/unannotated series metadata; missing derived "
            f"columns: {sorted(missing_meta)}. Run annotate(...) before pick_slots."
        )

    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None means "do not condition on weighting" - the public scheme,
            # where the single provided flag stands in for both axes at once.
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            # A slot with no series matching its predicate stays empty, and no substitute
            # is admitted from a neighbouring predicate. Relaxing the weighting to fill a
            # T1 slot would draw from the pool `SAG_FLUID_NOFS` selects from, since that
            # pool is what remains once the weighting is dropped: over the training corpus
            # it would put one series in two slots for 2383 of 4407 studies and leave 56%
            # of the T1 slot holding PD or T2. The presence mask would then assert a
            # sequence that was never acquired, and the per-diagnosis softmax of §6 would
            # divide its attention across two identical slots, giving one acquisition
            # about twice the weight it carries in a study that holds both. The mask is
            # there to say a slot is absent, which is what an absent slot is.
            if len(cand) == 0 and RULES["slot_fallback"] and fluid is False:
                # The relaxation the paragraph above rejects, reproduced because an
                # imported member was fitted with its T1 slots filled this way: over half
                # of that member's training studies had a T1 slot holding a series that
                # is not T1. Leaving those slots empty would present it with a presence
                # mask it never saw.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out

In [ ]:
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

# Series in which at least one sampled slice would not decode. A list rather than a
# counter because appending is atomic under the reader threads, and reported rather than
# swallowed: unreported, a decode failure is indistinguishable from a black knee.
DECODE_FAILED = []


def cache_tag(rules=None):
    """The name a decoded cache is stored under.

    It has to name everything that decides the pixels, not only their dimensions. Two
    configurations that agree on resolution, slice count, crop and band but disagree on
    how a slice is chosen produce different arrays of identical shape - so a tag built
    from the dimensions alone lets the second attach to the first one's file and train
    against pixels it never asked for, with nothing anywhere reporting a mismatch.

    A native reading keeps the plain name, so caches decoded before the rules existed
    stay valid; anything else earns a suffix.
    """
    r = dict(RULES if rules is None else rules)
    t = (f"{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_"
         f"{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}")
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += "_" + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t


def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))


def _order_dominant_axis(rec):
    """The slice order an imported member was fitted under.

    It sorts on the raw patient coordinate along whichever axis varies most across the
    stack, rather than on the projection onto the slice normal. The two differ by a sign,
    not by a formula: measured over this corpus every sagittal series has a slice normal
    with n_x in [-1.00, -0.98], so p.n is the negative of the raw x this sorts on and the
    two stacks come out exactly reversed. Because the band sampler truncates rather than
    rounds, its nine indices are not symmetric about the middle, so nine slices drawn from
    a twenty-six slice stack under one order share two with the other.

    Missing geometry falls back to `InstanceNumber` and then to a natural sort of the file
    name, both at the same 80% threshold the imported pipeline used.
    """
    files, d = rec["files"], rec["dir"]
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=["ImagePositionPatient", "InstanceNumber"])
            raw = getattr(ds, "ImagePositionPatient", None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, "InstanceNumber", None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))

    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare,
                                 r[2] if r[2] is not None else float("inf"), r[3]))
    elif sum(r[2] is not None for r in rows) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float("inf"), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return [r[0] for r in rows], True


def order_slices(rec):
    """Return the series' files sorted along the through-plane axis.

    A DICOM file name here is a SOP Instance UID, which is assigned arbitrarily. Sorting
    by it therefore produces an order uncorrelated with anatomy - measured over one
    series, Spearman between file-name rank and physical position is 0.009, i.e. none.
    Anything that assumes the file order means something is then operating on noise: the
    three channels of a "2.5D" input are three unrelated views rather than neighbouring
    slices, "the middle of the stack" is a random subset, and reversing slice order to
    normalise laterality reverses nothing meaningful.

    The physical order is recoverable exactly. Each slice carries its position in patient
    coordinates and the in-plane axes; projecting the position onto the slice normal
    gives a signed through-plane coordinate, monotonic along the stack:

        n = r_x  x  r_y ,      k = p . n

    `InstanceNumber` is the fallback. It usually tracks the projection up to sign, but
    interleaved and multi-echo acquisitions need not number slices in the order they
    occupy in space - but the projection is signed in patient
    coordinates, which is what laterality normalisation needs.
    """
    if RULES["order"] == "dominant_axis":
        return _order_dominant_axis(rec)
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None for k, _ in keyed):
        # A series with no usable geometry keeps its arbitrary order; that is worse than
        # sorting but better than dropping the series, and it is logged as a count.
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` physically spread slices from one series, at `out_size` pixels.

    Returns uint8 [n_slice, out, out] normalised per-series to its 1st-99th
    percentile. Percentiles rather than min/max because MR intensity has no absolute
    scale and a single bright vessel would otherwise compress the whole dynamic range.

    Reading is the expensive half of this pipeline, so the caller reads once at the
    largest configuration it needs and derives the smaller ones from the returned buffer
    rather than re-reading.
    """
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Spread the samples over a central band of the stack: the outermost slices of a knee
    # series are mostly soft tissue outside the joint. The band is a constant rather than
    # a literal because how much of the stack is worth reading depends on how many slices
    # are being taken - at three the middle is all that fits, while at sixteen the ends
    # are worth having, and a Baker cyst sits at the posteromedial end of a sagittal one.
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None                      # no shape is known here; see below
        planes.append(a)

    # A slice that would not decode has no shape of its own, and inventing one is how a
    # single unreadable file erases a whole series: a substitute allocated at the resize
    # target while the decoded slices are still native makes the shape check below take
    # the substitute as the authority and zero the good slices with it, leaving a black
    # slot that the presence mask still reports as acquired.
    #
    # A failure is instead filled from the nearest slice that did decode - the same
    # convention the sampler already uses when the band holds fewer distinct slices than
    # were asked for - and a series where nothing decodes is reported absent, which the
    # mask can express, rather than black, which it cannot.
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES["decode_fill"] == "zero":
        # What an imported member was fitted with: a failure becomes a zero plane at the
        # resize target, which the shape check below then propagates to the whole slot.
        # It is the behaviour the paragraph above describes and rejects, kept here only
        # because that member's weights were learned against slots blacked out this way.
        if not got:
            DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p
                  for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    # Slices of one series can still differ in matrix size - multi-echo and some
    # reformats do - and those are genuinely not stackable.
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # constant physical extent, then resize: PixelSpacing varies 3.4x across the corpus
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32. These buffers queue up between the reader threads and the
    # encoder, and at this size a float32 slot-series is several megabytes. Intensity is
    # already normalised into [0, 1] here, so eight bits cost nothing that a bilinear
    # resize has not already cost, and the queue is a quarter the size.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)

In [ ]:
def normalise_laterality(img, plane, lat):
    """Map every knee onto a left-knee convention.

    Coronal and axial views mirror under a horizontal flip. Sagittal stacks are not
    mirror images of each other - the slice order runs medial-to-lateral in opposite
    directions - so the channel order is reversed instead.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])

In [ ]:
# Where the geometric slice order may be remembered between runs. Unset on the platform,
# because each run gets a fresh machine and there is nothing to remember; set off it,
# where the same corpus is cached again at every resolution and slice count and the order
# is a function of neither. It is opt-in so that the scored run's behaviour is decided by
# the code rather than by whether a file happens to be lying about.
ORDER_CACHE = os.environ.get("RSNA_ORDER_CACHE") or None


def build_cache(slot_map, plane_map, lat_map, tag):
    """Decode every (study, slot) once into an in-memory uint8 array.

    Fine-tuning revisits the same pixels every epoch. Reading them from the mount each
    time would make the epoch count a function of I/O rather than of learning, so they
    are decoded once and held as bytes: intensity has already been normalised into
    [0, 1], and eight bits cost nothing a bilinear resize has not already cost.

    CACHE_SLICES positions are kept per slot, which the training loop reads as N_GROUP
    groups of GROUP consecutive channels.
    """
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    n_job = len(jobs)

    # Ordering first, and as its own pass. It reads one header per slice of every chosen
    # series - far more file opens than the decode that follows - and on a network mount
    # that is latency, not work, so it gets its own wider pool.
    t_ord = time.time()
    n_slice_total = sum(len(j[3]["files"]) for j in jobs)
    log(f"{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)")
    ok = done = 0
    CHUNK_O = 1024

    # A remembered order, when one is offered. The projection depends on the DICOM
    # geometry alone, so it is the same at every resolution and every slice count, and
    # it costs one header read per slice - the largest single cost in this pass. An entry
    # is validated by the number of files present, so a tree that has changed under it is
    # recomputed rather than trusted: order is derived data, and a stale entry would be
    # invisible in the way that matters most.
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec["SeriesInstanceUID"])
            if e and len(e["files"]) == len(rec["files"]):
                rec["ordered"] = e["files"]
                ok += int(e["good"])
                hit += 1
        jobs = [j for j in jobs if "ordered" not in j[3]]
        log(f"{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read")

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(
                    block, pool.map(lambda j: order_slices(j[3]), block)):
                rec["ordered"] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec["SeriesInstanceUID"]] = {"files": files, "good": bool(good)}
            # The ceiling is whichever comes first: the pass's own budget, or the share
            # of what is left of the run that it may take. The second is what makes the
            # first safe to set generously - a mount slow enough to matter cannot spend
            # the training time, because the budget shrinks as the run does.
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f"{tag}: ordering budget spent at {done}/{len(jobs)}; "
                    f"the rest keep file order")
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix(".tmp")
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f"{tag}: ordered {ok}/{n_job} by geometry "
        f"({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")
    n_failed_before = len(DECODE_FAILED)

    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(
                    block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > TIME_BUDGET:
                log(f"  {tag}: time budget reached during decode")
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f"{tag}: {int(mask.sum())}/{len(jobs)} slots filled"
        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))
    gc.collect()
    return studies, cache, mask

In [ ]:
class SlotHead(nn.Module):
    """Per-diagnosis attention over the slot embeddings of one study.

    Each finding is read on particular sequences - cruciates sagittally, collateral
    ligaments and the meniscal body coronally, patellar cartilage axially - so pooling
    the slots identically would dilute the one that carries the evidence with the rest.

    The aggregation is deliberately this simple. With a study-level label there is no
    signal telling the model which part of a study matters, so extra attention
    parameters below the slot level would have nothing to learn from and would spend
    their capacity fitting noise.
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        # An imported member carries a fixed per-(diagnosis, slot) tilt on the attention
        # logits, set from the anatomy table below rather than learned. It is a buffer, so
        # it travels in the state dict and must exist for that member to load; exp(0.55)
        # gives a preferred slot about 1.73x the weight of an unpreferred one, which
        # biases the softmax without ever excluding a slot.
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and n_out == len(TARGETS):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

In [ ]:
class Model(nn.Module):
    """Encoder plus head, trained end to end.

    A study arrives as a bag of slot images. The bag is flattened for the encoder and
    folded back before the head, so the encoder never sees the study structure and the
    head never sees pixels.
    """

    def __init__(self, backbone, dim, pool="cls_mean", prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            # The cache is held at the highest resolution any configuration needs; the
            # rest downsample from it, so every configuration sees the same pixels
            # through a different sampling grid rather than a different crop.
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear",
                              align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            # The upper tail of each channel over the patch grid, taken per channel
            # rather than by selecting whole patches: a finding occupies a small part of
            # the field, so a plain mean over 256 patches dilutes it by two orders of
            # magnitude, and this keeps the top eighth of each channel's responses.
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

In [ ]:
def build_model(unfreeze_last, source=None, variant="small", pool="cls_mean", prior=False):
    from transformers import AutoModel

    p = Path(source) if source is not None else DINO
    if p is None:
        raise FileNotFoundError("DINOv2 weights not attached")

    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)

    for prm in bb.parameters():
        prm.requires_grad = False

    n_unfreeze = max(0, int(unfreeze_last))
    if n_unfreeze > 0:
        for blk in bb.encoder.layer[max(0, n_layer - n_unfreeze):]:
            for prm in blk.parameters():
                prm.requires_grad = True
        # LayerNorm is only trainable when backbone fine-tuning is requested.
        for prm in bb.layernorm.parameters():
            prm.requires_grad = True

    dim = bb.config.hidden_size
    trainable = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    log(
        f"backbone: {n_layer} blocks, last {n_unfreeze} trainable "
        f"({trainable / 1e6:.1f}M backbone params)"
    )
    return Model(bb, dim, pool=pool, prior=prior)

In [ ]:
FINGERPRINT_TOL = 2e-3

def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed

    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(
        0, 256, (2, n_slot, group, img_size, img_size),
        generator=g, dtype=torch.uint8
    ).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0

    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out


def check_fingerprint(
    model,
    dev,
    img_size,
    expected,
    tol=FINGERPRINT_TOL,
    tag="",
    seed=None,
):
    # IMPORTANT: a stored fingerprint must be reproduced with the same
    # deterministic synthetic-input seed that was used when it was saved.
    got = fingerprint(model, dev, img_size, seed=seed)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f"{tag}fingerprint shape {got.shape} != stored {exp.shape}")
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(
            f"{tag}fingerprint differs by {d:.4g} > {tol:g}; "
            "architecture/preprocessing contract moved"
        )
    log(f"{tag}fingerprint matches within {d:.2g} (seed={seed})")
    return d


def window_starts(n_slice, group):
    if n_slice >= group:
        return list(range(n_slice - group + 1))
    return [0]


def _apply_frontier_pool(probs):
    v = probs.mean(dim=0)
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    for target, mode in FRONTIER_TARGET_POOL.items():
        j = target_idx[target]
        x = probs[:, :, j]
        if mode == "max":
            v[:, j] = x.max(dim=0).values
        elif mode.startswith("top"):
            k = min(int(mode[3:]), x.shape[0])
            v[:, j] = x.topk(k, dim=0).values.mean(dim=0)
        elif mode == "mean":
            v[:, j] = x.mean(dim=0)
        else:
            raise ValueError(mode)
    return v


@torch.no_grad()
def predict_member_frontier(model, cache, mask, idx, dev, img_size):
    starts = window_starts(cache.shape[2], GROUP)
    model.eval()
    pred = np.empty((len(idx), len(TARGETS)), np.float32)

    for b0 in range(0, len(idx), EVAL_BATCH):
        sel = np.asarray(idx[b0:b0 + EVAL_BATCH], dtype=int)
        b1 = b0 + len(sel)
        m = torch.from_numpy(mask[sel]).to(dev)
        win = []

        for st in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(cache[sel, :, st:st + GROUP])
            ).to(dev)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                logits = model(rows, m, img_size).float()
            win.append(torch.sigmoid(logits))

        probs = torch.stack(win, dim=0)
        pred[b0:b1] = _apply_frontier_pool(probs).cpu().numpy()

    return pred

## 5. Final test metadata and package preflight

In [ ]:
test_df = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})
test_series = pd.read_csv(
    ROOT / "test_series.csv",
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
)

if "StudyInstanceUID" not in test_df.columns:
    raise RuntimeError("test.csv has no StudyInstanceUID")

test_ids = test_df["StudyInstanceUID"].astype(str).tolist()
if len(test_ids) != len(set(test_ids)):
    raise RuntimeError("test.csv contains duplicate StudyInstanceUID")

plane_map = dict(zip(
    test_series["SeriesInstanceUID"].astype(str),
    test_series["Anatomical_Plane"],
))

# walk_selected() returns raw DICOM header metadata. The checkpoint slot
# contract requires the derived columns produced by annotate().
hte_raw = walk_selected("test_series", test_ids)
hte = annotate(hte_raw)

required_derived = {"fatsat", "fluid", "weight", "px"}
missing_derived = required_derived.difference(hte.columns)
if missing_derived:
    raise RuntimeError(
        f"test metadata annotation failed; missing columns: {sorted(missing_derived)}"
    )

covered_test_ids = set(hte["StudyInstanceUID"].astype(str))
missing_header_studies = sorted(set(test_ids) - covered_test_ids)
if missing_header_studies:
    raise RuntimeError(
        "test header pass did not cover every test study; "
        f"missing {len(missing_header_studies)} "
        f"(first={missing_header_studies[:3]})"
    )

log(
    f"test header+annotation pass: {len(hte)} series across "
    f"{hte['StudyInstanceUID'].nunique()} studies"
)

# Audit only. The model continues to use the same DICOM-header inference used
# throughout the successful training/fine-tuning pipeline.
audit_official_sequence_metadata(hte, test_series)

print(
    "Derived metadata columns ready:",
    {c: str(hte[c].dtype) for c in ("fatsat", "fluid", "weight", "px")}
)

branch_packages = {
    "BASE": {
        "path": BASE_WEIGHTS,
        "manifest": BASE_MANIFEST,
    }
}
if NEED_FTC:
    branch_packages["FTC"] = {
        "path": STAGE4,
        "manifest": STAGE4_MANIFEST,
    }

# Exact preflight before touching pixels.
for branch, pack in branch_packages.items():
    members = pack["manifest"]["members"]
    missing = [
        m["file"] for m in members
        if not (pack["path"] / str(m["file"])).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"{branch} package incomplete; first missing checkpoint: {missing[0]}"
        )
    log(f"{branch}: {len(members)} complete checkpoint(s)")

# Union pixel groups so the DICOM cache is decoded only once per unique configuration.
pixel_groups = {}
for branch, pack in branch_packages.items():
    for m in pack["manifest"]["members"]:
        pixel_groups.setdefault(m["pixel_group"], []).append(
            (branch, pack["path"], m)
        )

print("Unique test pixel groups:", len(pixel_groups))
for key, items in pixel_groups.items():
    cfg = json.loads(key)
    counts = {}
    for b, _, _ in items:
        counts[b] = counts.get(b, 0) + 1
    print(
        f"- {cfg['img']}px / {cfg['slices']} slices / group={cfg['group']} "
        f"/ crop={cfg['crop_mm']}mm -> {counts}"
    )

## 6. Final inference utilities

In [ ]:
def _rank_matrix(x):
    return (
        pd.DataFrame(np.asarray(x))
        .rank(method="average", pct=True)
        .to_numpy(np.float64)
    )


def centered_window_subset(starts, n_keep):
    starts = list(starts)
    n_keep = max(1, min(int(n_keep), len(starts)))
    if n_keep >= len(starts):
        return starts
    mid = (len(starts) - n_keep) // 2
    return starts[mid:mid + n_keep]


@torch.no_grad()
def predict_member_final(model, cache, mask, idx, dev, img_size, starts):
    model.eval()
    pred = np.empty((len(idx), len(TARGETS)), np.float32)

    for b0 in range(0, len(idx), EVAL_BATCH):
        sel = np.asarray(idx[b0:b0 + EVAL_BATCH], dtype=int)
        b1 = b0 + len(sel)
        m = torch.from_numpy(mask[sel]).to(dev)
        wins = []

        for st in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(cache[sel, :, st:st + GROUP])
            ).to(dev)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                logits = model(rows, m, img_size).float()
            wins.append(torch.sigmoid(logits))

        probs = torch.stack(wins, dim=0)
        pred[b0:b1] = _apply_frontier_pool(probs).cpu().numpy()

    return pred


def aggregate_member_ranks(per_member, test_ids):
    if not per_member:
        raise RuntimeError("branch has no member predictions")

    pos = {s: i for i, s in enumerate(test_ids)}
    M, N, T = len(per_member), len(test_ids), len(TARGETS)
    ranks = np.full((M, N, T), np.nan, np.float64)

    for mi, item in enumerate(per_member):
        ids = item["ids"]
        rows = [pos[s] for s in ids if s in pos]
        if len(rows) != len(ids):
            raise RuntimeError(f"{item['id']}: prediction contains unexpected StudyInstanceUID")
        if len(ids) != len(test_ids) or set(ids) != set(test_ids):
            raise RuntimeError(
                f"{item['id']}: member did not cover all hidden test studies"
            )
        ranks[mi, rows] = _rank_matrix(item["pred"])

    if np.isnan(ranks).any():
        raise RuntimeError("non-finite or incomplete member-rank tensor")

    return ranks.mean(axis=0)


def validate_final_submission(frame, test_df):
    expected = ["StudyInstanceUID", *TARGETS]
    if frame.columns.tolist() != expected:
        raise RuntimeError(
            f"submission schema mismatch: {frame.columns.tolist()} != {expected}"
        )
    if len(frame) != len(test_df):
        raise RuntimeError("submission row count mismatch")
    if frame["StudyInstanceUID"].astype(str).tolist() != test_df["StudyInstanceUID"].astype(str).tolist():
        raise RuntimeError("submission StudyInstanceUID order mismatch")
    if frame["StudyInstanceUID"].duplicated().any():
        raise RuntimeError("duplicate StudyInstanceUID in submission")
    arr = frame[TARGETS].to_numpy(np.float64)
    if not np.isfinite(arr).all():
        raise RuntimeError("non-finite value in submission")
    if (arr < 0).any() or (arr > 1).any():
        raise RuntimeError("submission values outside [0,1]")
    return frame

## 7. Run BASE and, only if selected, FT-C hidden-test inference

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for final inference")
dev = torch.device("cuda")
log(f"device: {dev}")

per_branch = {name: [] for name in branch_packages}

# Used only for adaptive runtime protection.
model_total = sum(len(p["manifest"]["members"]) for p in branch_packages.values())
model_done = 0
observed_fixed = None
observed_per_window = None

for gi, (pixel_key, items) in enumerate(pixel_groups.items(), 1):
    cfg = json.loads(pixel_key)
    adopt_config_globals(cfg)

    log(
        f"decode group {gi}/{len(pixel_groups)}: {IMG}px, "
        f"{CACHE_SLICES} slices, group={GROUP}, crop={CROP_MM}mm"
    )

    slot_map = pick_slots(hte, plane_map)

    # Cheap metadata preflight before expensive DICOM pixel decoding.
    slot_counts = np.asarray(
        [len(slot_map.get(st, {})) for st in test_ids],
        dtype=int,
    )
    if len(slot_counts) != len(test_ids):
        raise RuntimeError("slot-map study count mismatch")
    if (slot_counts == 0).any():
        zero_ids = [st for st, n in zip(test_ids, slot_counts) if n == 0]
        log(
            f"WARNING: {len(zero_ids)} test study/studies have zero matching "
            f"slots for pixel group {gi}; first={zero_ids[:3]}"
        )
    log(
        f"g{gi} slot preflight: mean={slot_counts.mean():.2f}/{N_SLOT}, "
        f"min={slot_counts.min()}, max={slot_counts.max()}"
    )

    st_te, Cte, Mte = build_cache(
        slot_map,
        plane_map,
        lat_of(hte, f"test g{gi} "),
        f"test g{gi}",
    )

    if set(st_te) != set(test_ids):
        missing = sorted(set(test_ids) - set(st_te))
        raise RuntimeError(
            f"pixel group {gi} did not produce all test studies; "
            f"missing {len(missing)} (first={missing[:3]})"
        )

    idx = np.arange(len(st_te))
    all_starts = window_starts(Cte.shape[2], GROUP)

    # Put the 20-member BASE safety family first.
    items = sorted(
        items,
        key=lambda x: (
            0 if x[0] == "BASE" else 1,
            int(x[2].get("fold", 99)),
            -(float(x[2].get("holdout") or 0.0)),
        ),
    )

    for branch, pkg_path, m in items:
        model_done += 1
        remaining_after = model_total - model_done
        left = TIME_BUDGET - (time.time() - T0)

        if left <= 0:
            raise RuntimeError("final inference time budget exhausted")

        use_starts = list(all_starts)

        if observed_fixed is not None and observed_per_window is not None:
            remaining_including_current = remaining_after + 1
            projected_each = observed_fixed + len(use_starts) * observed_per_window
            projected_total = projected_each * remaining_including_current

            if projected_total > left * 0.92:
                room_each = left * 0.90 / max(remaining_including_current, 1)
                n_win = int(
                    (room_each - observed_fixed) / max(observed_per_window, 1e-9)
                )
                n_win = max(MIN_WINDOWS, min(len(use_starts), n_win))
                use_starts = centered_window_subset(use_starts, n_win)
                log(
                    f"{left/60:.0f} min left: runtime guard uses "
                    f"{len(use_starts)}/{len(all_starts)} windows for {branch}:{m['id']}"
                )

        ck_path = Path(pkg_path) / str(m["file"])
        t_load = time.time()
        ck = torch.load(ck_path, map_location="cpu", weights_only=False)

        member_cfg = m.get("config") or {}
        model = build_model(
            0,  # inference only; requires_grad state does not affect the forward pass
            source=DINO,
            variant=member_cfg.get("variant", "small"),
            pool=member_cfg.get("pool", "cls_mean"),
            prior=bool(member_cfg.get("prior", False)),
        ).to(dev)
        model.load_state_dict(ck["model"], strict=True)

        fp_seed = int(ck.get("fingerprint_seed", 2026))
        if "fingerprint" not in ck:
            raise WeightsError(f"{branch}:{m['id']} has no checkpoint fingerprint")
        check_fingerprint(
            model,
            dev,
            IMG,
            ck["fingerprint"],
            tag=f"{branch}:{m['id']}: ",
            seed=fp_seed,
        )
        t_ready = time.time()

        pred = predict_member_final(
            model, Cte, Mte, idx, dev, IMG, starts=use_starts
        )
        t_done = time.time()

        per_branch[branch].append({
            "id": m["id"],
            "fold": m.get("fold"),
            "ids": list(st_te),
            "pred": pred,
            "n_windows": len(use_starts),
        })

        observed_fixed = t_ready - t_load
        observed_per_window = (t_done - t_ready) / max(len(use_starts), 1)

        log(
            f"[{model_done}/{model_total}] {branch}:{m['id']} "
            f"fold={m.get('fold')} windows={len(use_starts)} "
            f"time={t_done - t_load:.0f}s"
        )

        del model, ck, pred
        gc.collect()
        torch.cuda.empty_cache()

    del Cte, Mte
    gc.collect()
    torch.cuda.empty_cache()

diag_rows = []
for branch, items in per_branch.items():
    for x in items:
        diag_rows.append({
            "branch": branch,
            "member": x["id"],
            "fold": x.get("fold"),
            "n_windows": x["n_windows"],
        })
diag = pd.DataFrame(diag_rows)
diag.to_csv(DIAG_DIR / "final_inference_members.csv", index=False)
display(diag)

## 8. Build exactly one final `submission.csv`

In [ ]:
base_rank = aggregate_member_ranks(per_branch["BASE"], test_ids)

if NEED_FTC:
    ftc_rank = aggregate_member_ranks(per_branch["FTC"], test_ids)
else:
    ftc_rank = base_rank.copy()

final_score = base_rank.copy()

for j, t in enumerate(TARGETS):
    a = float(FINAL_ALPHAS[t])
    final_score[:, j] = (
        (1.0 - a) * base_rank[:, j]
        + a * ftc_rank[:, j]
    )

# Final rank normalisation: the competition pipeline is rank-ensemble based,
# so all target columns finish on the same [0,1] percentile scale.
final_rank = _rank_matrix(final_score)

pred_frame = pd.DataFrame(final_rank, columns=TARGETS)
pred_frame.insert(0, "StudyInstanceUID", test_ids)

# The cache order may be sorted while test.csv has its own official order.
submission = (
    test_df[["StudyInstanceUID"]]
    .merge(pred_frame, on="StudyInstanceUID", how="left")
)

if submission[TARGETS].isna().any().any():
    raise RuntimeError("final inference missed one or more hidden test studies")

submission = validate_final_submission(submission, test_df)
submission.to_csv(FINAL_PATH, index=False)

# Validate the exact bytes/table that will be submitted, not only the in-memory frame.
submission_disk = pd.read_csv(
    FINAL_PATH, dtype={"StudyInstanceUID": str}
)
submission_disk = validate_final_submission(
    submission_disk,
    test_df.astype({"StudyInstanceUID": str}),
)
if FINAL_PATH.stat().st_size <= 0:
    raise RuntimeError("submission.csv was created but is empty")

# A small audit file, but NO second submission candidate.
final_status = {
    "final_ok": True,
    "submission_path": str(FINAL_PATH),
    "rows": int(len(submission)),
    "targets": TARGETS,
    "base_members_used": int(len(per_branch["BASE"])),
    "ftc_members_used": int(len(per_branch.get("FTC", []))),
    "ftc_inference_required": bool(NEED_FTC),
    "target_ftc_alphas": {k: float(v) for k, v in FINAL_ALPHAS.items()},
    "selection_rule": "nested expert-CV conservative rank blend; BASE fallback",
}
(DIAG_DIR / "final_status.json").write_text(json.dumps(final_status, indent=2))

print("=" * 72)
print("FINAL SUBMISSION READY")
print("Submit this file and only this file:")
print(FINAL_PATH)
print("=" * 72)
print(json.dumps(final_status, indent=2))
display(submission.head())

## Final output

After a successful `Save Version / Run All`, submit:

**`submission.csv`**

Do not submit files inside `final_diagnostics/`; they are audit metadata only.